# 002 · Comparison and Logical Operators

> **Syntax › Operators** · Python 3.14

Comparisons in Python read almost like maths — `0 < x < 10` just works — and the logical operators are spelled as English words. More surprisingly, `and` and `or` don't return `True`/`False`: they return one of their **operands**.

**In this notebook you will learn:**

- the comparison operators and how `==` differs from C#'s reference equality
- chained comparisons such as `0 < x < 10`
- `and`, `or`, `not` vs `&&`, `||`, `!`, including short-circuiting
- why `name or "default"` works (and when it's not a safe replacement for `??`)
- the conditional expression `a if cond else b` vs C#'s `?:`
- what happens when you compare values of different types

## C# mental model

| C# | Python | Note |
|---|---|---|
| `==`, `!=` | `==`, `!=` | Python `==` compares **values** for all built-in types (lists, dicts, …) |
| `<`, `>`, `<=`, `>=` | `<`, `>`, `<=`, `>=` | also work on strings, lists and tuples |
| `x > 0 && x < 10` | `0 < x < 10` | chained comparisons |
| `&&`, `\|\|`, `!` | `and`, `or`, `not` | keywords, not symbols |
| `a ?? b` | `a if a is not None else b` | `a or b` is close, but not equivalent |
| `cond ? a : b` | `a if cond else b` | different order! |
| `ReferenceEquals(a, b)` | `a is b` | see *Operators › 004 · Membership and Identity Operators* |

## Comparison operators

Comparisons return `True` or `False` (capitalised — they are the two `bool` constants).

In [ ]:
x = 7
print(x == 7, x != 7)
print(x < 10, x <= 7, x > 10, x >= 8)

`==` is **value equality** for every built-in type — no `SequenceEqual` or `Equals` overrides needed. Numeric types compare by mathematical value across types.

```csharp
// C#
var a = new List<int> { 1, 2 };
var b = new List<int> { 1, 2 };
Console.WriteLine(a == b);              // False — reference comparison
Console.WriteLine(a.SequenceEqual(b));  // True
```

In [ ]:
print([1, 2] == [1, 2])              # True — element by element
print({"a": 1} == {"a": 1})          # True
print(1 == 1.0, 1 == True)           # True True — numbers compare by value (bool is a subclass of int)
print("abc" == "ABC")                # False — case-sensitive (ordinal)

Ordering works on strings (by Unicode code point, like `string.CompareOrdinal`) and on sequences (lexicographically, item by item):

In [ ]:
print("apple" < "banana")       # True
print("Zebra" < "apple")        # True — uppercase letters sort before lowercase
print((1, 2, 3) < (1, 3))       # True — decided at the second item
print([2] > [1, 99, 99])        # True — decided at the first item

Your own classes control `==` and `<` by implementing `__eq__`, `__lt__` and friends — the equivalent of overriding `Equals` and implementing `IComparable<T>` *(see Classes & OOP › 007 · Dunder Methods)*.

## Chained comparisons

Python lets you chain comparisons the way you would write them in maths. `a < b < c` means `a < b and b < c`, except that `b` is evaluated **only once**.

```csharp
// C#
if (x > 0 && x < 10) { ... }
if (x is > 0 and < 10) { ... }   // C# 9 pattern alternative
```

In [ ]:
x = 5
print(0 < x < 10)            # True
print(0 < x < 3)             # False
print(1 <= x <= 5 <= 100)    # chains can be any length

a = b = c = 3
print(a == b == c)           # all three equal

⚠️ **Gotcha:** chaining works with *any* comparison operators, which can produce legal but confusing code. `a < b > c` means `a < b and b > c` — it says nothing about how `a` relates to `c`. Keep chains in one direction.

In [ ]:
print(1 < 3 > 2)          # True — but 1 and 2 are never compared
print(False == False in [False])   # True: means (False == False) and (False in [False])

## Logical operators: `and`, `or`, `not`

The logical operators are keywords. They short-circuit exactly like `&&` and `||`: the right operand is evaluated only if needed.

```csharp
// C#
if (age >= 18 && hasTicket) { ... }
if (!isBanned || isAdmin) { ... }
```

In [ ]:
age, has_ticket, is_banned, is_admin = 21, True, False, False

print(age >= 18 and has_ticket)
print(not is_banned or is_admin)
print(not (age < 18))

Precedence: `not` binds tighter than `and`, which binds tighter than `or` — the same as `!`, `&&`, `||` in C#. And all three bind more loosely than comparisons, so `x > 0 and x < 10` needs no parentheses.

To see short-circuiting in action, use a helper that reports when it is evaluated:

In [ ]:
def check(label, result):
    print(f"evaluated {label}")
    return result

print(check("left", False) and check("right", True))   # right side never runs
print(check("left", True) or check("right", True))     # right side never runs

⚠️ **Gotcha:** `&` and `|` are *bitwise* operators, not logical ones *(see Operators › 003 · Bitwise Operators)*. They don't short-circuit, and they bind more tightly than comparisons — `x > 0 & x < 10` means `x > (0 & x) < 10`. Always use `and` / `or` for boolean logic.

## `and` / `or` return operand values

Here is the big difference from C#. `&&` and `||` always produce a `bool`. Python's `and` and `or` return **one of the operands** — the one that decided the outcome:

- `a or b` → `a` if `a` is truthy, otherwise `b`
- `a and b` → `a` if `a` is falsy, otherwise `b`

"Truthy" and "falsy" means how a value behaves in a boolean context: `0`, `""`, `None`, empty collections and `False` are falsy; almost everything else is truthy *(see Data Types › 004 · Booleans and Truthiness)*.

In [ ]:
print(0 or 42)           # 42   — 0 is falsy, so the right operand is returned
print("alice" or "anon") # alice
print([] or [1, 2])      # [1, 2]
print(3 and 5)           # 5    — 3 is truthy, so the right operand decides
print(0 and 5)           # 0    — 0 is falsy, returned without evaluating 5
print(repr(None or ""))  # ''   — both falsy: the last one is returned

`not` is the exception — it always returns a real `bool`:

In [ ]:
print(not 0, not "text", not [])

## `or` as a default value — vs `??`

This makes `value or default` a very common idiom for "use a fallback if the value is missing or empty":

```csharp
// C#
string displayName = name ?? "Anonymous";
```

In [ ]:
def greet(name=None):
    display_name = name or "Anonymous"
    print(f"Hello, {display_name}!")

greet("alice")
greet(None)
greet("")          # empty string is falsy too -> Anonymous

⚠️ **Gotcha:** `or` is **not** the same as `??`. C#'s `??` only replaces `null`; Python's `or` replaces **every falsy value** — including `0`, `0.0`, `""`, `[]` and `False`, which are often perfectly valid:

In [ ]:
quantity = 0              # a legitimate value!

print(quantity or 10)                              # 10 — wrong, 0 was replaced
print(quantity if quantity is not None else 10)    # 0  — true ?? semantics

Python has no `??`, `?.` or `??=` operators. Use `x if x is not None else default` when `0`/`""`/`[]` are valid values, and `or` when any falsy value should fall back *(see Data Types › 005 · None)*.

## Conditional expression: `a if cond else b`

Python's version of the ternary operator `?:` reads like English — and puts the condition **in the middle**:

```csharp
// C#
var label = age >= 18 ? "adult" : "minor";
```

In [ ]:
age = 15
label = "adult" if age >= 18 else "minor"
print(label)

scores = [72, 45, 90]
print(["pass" if s >= 50 else "fail" for s in scores])   # handy inside comprehensions

Only the selected branch is evaluated, just like `?:`. They can be nested, but that quickly becomes hard to read — prefer `if`/`elif` statements or a `match` for more than two outcomes *(see Control Flow › 001 · if / elif / else)*.

In [ ]:
n = 0
sign = "positive" if n > 0 else "negative" if n < 0 else "zero"   # legal, but don't overdo it
print(sign)

## Comparing different types

`==` between unrelated types is always allowed and simply returns `False` — there's no compile error like C#'s *"operator '==' cannot be applied to operands of type 'int' and 'string'"*.

In [ ]:
print(1 == "1")        # False — no implicit conversion (unlike JavaScript)
print([1] == (1,))     # False — list vs tuple
print(None == 0)       # False

Ordering comparisons (`<`, `>`, …) between unrelated types raise a `TypeError` at runtime. (Python 2 allowed this and gave arbitrary results; Python 3 refuses.)

In [ ]:
for left, right in [(1, "a"), (None, 0), ([1], (1,))]:
    try:
        print(left < right)
    except TypeError as e:
        print(f"{type(e).__name__}: {e}")

This matters when sorting: a list that mixes numbers and strings cannot be sorted without a `key=` function.

In [ ]:
mixed = [3, "10", 2]
try:
    sorted(mixed)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(sorted(mixed, key=int))   # compare by a common key instead

## Summary

| Task | C# | Python |
|---|---|---|
| Value equality | `a.Equals(b)` / `SequenceEqual` | `a == b` |
| Range check | `x > 0 && x < 10` | `0 < x < 10` |
| Logical AND / OR / NOT | `&&` / `\|\|` / `!` | `and` / `or` / `not` |
| Null-coalescing | `a ?? b` | `a if a is not None else b` |
| Falsy fallback | — | `a or b` |
| Ternary | `c ? a : b` | `a if c else b` |
| Compare unrelated types | compile error | `==` → `False`; `<` → `TypeError` |

## Exercises

1. Write `in_range(value, low, high)` that returns `True` when `low <= value <= high`, using a chained comparison.
2. Predict the output of each line, then check: `print("" or None or 0)`, `print(1 and "yes" and [])`, `print(not None)`.
3. Write `display(count)` that returns `count` if it isn't `None`, otherwise `"n/a"` — make sure `display(0)` returns `0`.
4. Rewrite this C# using a Python conditional expression: `var parity = n % 2 == 0 ? "even" : "odd";`

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def in_range(value, low, high):
    return low <= value <= high

print(in_range(5, 1, 10), in_range(11, 1, 10))   # True False

# 2
print("" or None or 0)     # 0  — all falsy, so the last operand is returned
print(1 and "yes" and [])  # [] — all truthy until [], which is returned
print(not None)            # True

# 3 — `count or "n/a"` would be wrong for 0
def display(count):
    return count if count is not None else "n/a"

print(display(0), display(None))   # 0 n/a

# 4
n = 7
parity = "even" if n % 2 == 0 else "odd"
print(parity)
```

</details>